## Dataset Feature Inventory

Inspect the processed customer dataset before selecting features for customer segmentation.

In [1]:
import pandas as pd
from pathlib import Path

processed_dataset_path = Path(
    "../data/processed/processed_dataset.csv"
)

processed_dataset = pd.read_csv(
    processed_dataset_path
)

print(f"Dataset Shape: {processed_dataset.shape}")

print("\nColumns:")
print(processed_dataset.columns.tolist())

print("\nData Types:")
print(processed_dataset.dtypes)

Dataset Shape: (96096, 5)

Columns:
['customer_unique_id', 'total_orders', 'total_spent', 'average_order_value', 'average_review_score']

Data Types:
customer_unique_id          str
total_orders              int64
total_spent             float64
average_order_value     float64
average_review_score    float64
dtype: object


## Segmentation Feature Selection

Select customer behavior, spending, and satisfaction features for clustering while excluding the customer identifier.

In [2]:
segmentation_features = [
    "total_orders",
    "total_spent",
    "average_order_value",
    "average_review_score"
]

customer_segmentation_data = processed_dataset[
    segmentation_features
].copy()

print("Selected Features:")
print(segmentation_features)

print(
    f"\nSegmentation Data Shape: "
    f"{customer_segmentation_data.shape}"
)

print("\nSelected Feature Data Types:")
print(customer_segmentation_data.dtypes)

Selected Features:
['total_orders', 'total_spent', 'average_order_value', 'average_review_score']

Segmentation Data Shape: (96096, 4)

Selected Feature Data Types:
total_orders              int64
total_spent             float64
average_order_value     float64
average_review_score    float64
dtype: object


## Clustering Preprocessing

Prepare the selected segmentation features for clustering by handling missing values and standardizing their numerical scale.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Handle missing values using feature-wise median imputation
imputer = SimpleImputer(strategy="median")

imputed_segmentation_data = imputer.fit_transform(
    customer_segmentation_data
)

# Standardize features for distance-based clustering
scaler = StandardScaler()

scaled_segmentation_data = scaler.fit_transform(
    imputed_segmentation_data
)

print(
    f"Imputed Data Shape: "
    f"{imputed_segmentation_data.shape}"
)

print(
    f"Scaled Data Shape: "
    f"{scaled_segmentation_data.shape}"
)

print(
    f"Remaining NaN Values: "
    f"{pd.isna(scaled_segmentation_data).sum()}"
)

Imputed Data Shape: (96096, 4)
Scaled Data Shape: (96096, 4)
Remaining NaN Values: 0


## K-Means Cluster Selection

Evaluate multiple cluster counts using inertia and silhouette score to identify an appropriate number of customer segments.

In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

cluster_results = []

# Use a representative sample for efficient model selection
sample_size = min(10000, len(scaled_segmentation_data))

sample_data = pd.DataFrame(
    scaled_segmentation_data
).sample(
    n=sample_size,
    random_state=42
).values

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(sample_data)

    inertia = kmeans.inertia_

    silhouette = silhouette_score(
        sample_data,
        labels
    )

    cluster_results.append({
        "n_clusters": k,
        "inertia": inertia,
        "silhouette_score": silhouette
    })

cluster_results = pd.DataFrame(
    cluster_results
)

print(cluster_results)

   n_clusters       inertia  silhouette_score
0           2  26919.203750          0.554998
1           3  19441.455847          0.600887
2           4  12735.856054          0.620453
3           5   9727.333390          0.626089
4           6   7831.929812          0.632181
5           7   6537.527685          0.558541
6           8   5723.391689          0.575461


## Clustering Algorithm Comparison

Compare multiple clustering algorithms using consistent evaluation metrics before selecting the final segmentation model.

In [7]:
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# Use a fixed representative sample for efficient model comparison
sample_size = min(10000, len(scaled_segmentation_data))

model_selection_sample = pd.DataFrame(
    scaled_segmentation_data
).sample(
    n=sample_size,
    random_state=42
).values

comparison_results = []

models = {
    "K-Means": KMeans(
        n_clusters=6,
        random_state=42,
        n_init=10
    ),
    "MiniBatch K-Means": MiniBatchKMeans(
        n_clusters=6,
        random_state=42,
        n_init=10,
        batch_size=1024
    ),
    "Gaussian Mixture": GaussianMixture(
        n_components=6,
        random_state=42
    )
}

for model_name, model in models.items():

    model.fit(model_selection_sample)

    labels = model.predict(model_selection_sample)

    silhouette = silhouette_score(
        model_selection_sample,
        labels
    )

    davies_bouldin = davies_bouldin_score(
        model_selection_sample,
        labels
    )

    comparison_results.append({
        "Model": model_name,
        "Silhouette Score": silhouette,
        "Davies-Bouldin Index": davies_bouldin
    })

model_comparison = pd.DataFrame(
    comparison_results
)

print(model_comparison)

               Model  Silhouette Score  Davies-Bouldin Index
0            K-Means          0.632181              0.655880
1  MiniBatch K-Means          0.481821              0.923802
2   Gaussian Mixture          0.173988              2.542098


## Final Training Data

Prepare the complete customer-level feature matrix using the locked segmentation features and preserve customer identifiers separately.

In [3]:
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load the final processed dataset
processed_dataset_path = Path(
    "../data/processed/processed_dataset.csv"
)

processed_dataset = pd.read_csv(
    processed_dataset_path
)

# Locked segmentation feature set
segmentation_features = [
    "total_orders",
    "total_spent",
    "average_order_value",
    "average_review_score"
]

# Preserve customer identifiers separately
customer_ids = processed_dataset[
    "customer_unique_id"
].copy()

# Select model features
training_features = processed_dataset[
    segmentation_features
].copy()

# Handle missing values
final_imputer = SimpleImputer(
    strategy="median"
)

imputed_training_features = final_imputer.fit_transform(
    training_features
)

# Standardize features for distance-based clustering
final_scaler = StandardScaler()

scaled_training_features = final_scaler.fit_transform(
    imputed_training_features
)

print(f"Customer Records: {len(customer_ids)}")
print(
    f"Training Feature Shape: "
    f"{training_features.shape}"
)
print(
    f"Scaled Feature Shape: "
    f"{scaled_training_features.shape}"
)

print(
    f"\nRemaining NaN Values: "
    f"{pd.isna(scaled_training_features).sum()}"
)

print("\nTraining Features:")
print(training_features.columns.tolist())

Customer Records: 96096
Training Feature Shape: (96096, 4)
Scaled Feature Shape: (96096, 4)

Remaining NaN Values: 0

Training Features:
['total_orders', 'total_spent', 'average_order_value', 'average_review_score']


## Train Customer Segmentation Model

Train the final K-Means segmentation pipeline using the complete customer-level feature dataset and the selected six-cluster configuration.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Final Customer Segmentation pipeline
segmentation_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KMeans(
                n_clusters=6,
                random_state=42,
                n_init=10
            )
        )
    ]
)

# Train on the complete customer dataset
segmentation_pipeline.fit(
    training_features
)

# Generate cluster assignments
cluster_labels = segmentation_pipeline.predict(
    training_features
)

print("Customer Segmentation Model trained successfully.")

print(
    f"\nTotal Customers: "
    f"{len(cluster_labels)}"
)

print(
    f"Number of Clusters: "
    f"{segmentation_pipeline.named_steps['model'].n_clusters}"
)

print("\nCluster Distribution:")
print(
    pd.Series(
        cluster_labels,
        name="cluster"
    ).value_counts().sort_index()
)

Customer Segmentation Model trained successfully.

Total Customers: 96096
Number of Clusters: 6

Cluster Distribution:
cluster
0    69743
1     2973
2    20488
3     2848
4       43
5        1
Name: count, dtype: int64


## Evaluate Customer Segmentation

Evaluate the trained segmentation model using clustering metrics and inspect the distribution of customers across the generated clusters.

In [5]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# Use a fixed sample for computationally efficient evaluation
evaluation_sample_size = min(
    10000,
    len(scaled_training_features)
)

evaluation_indices = pd.Series(
    range(len(scaled_training_features))
).sample(
    n=evaluation_sample_size,
    random_state=42
).values

evaluation_data = scaled_training_features[
    evaluation_indices
]

evaluation_labels = cluster_labels[
    evaluation_indices
]

# Calculate clustering evaluation metrics
silhouette = silhouette_score(
    evaluation_data,
    evaluation_labels
)

davies_bouldin = davies_bouldin_score(
    evaluation_data,
    evaluation_labels
)

# Full cluster distribution
cluster_distribution = (
    pd.Series(
        cluster_labels,
        name="cluster"
    )
    .value_counts()
    .sort_index()
)

cluster_percentages = (
    cluster_distribution
    / len(cluster_labels)
    * 100
)

evaluation_summary = pd.DataFrame({
    "Customer Count": cluster_distribution,
    "Percentage": cluster_percentages.round(2)
})

print("Customer Segmentation Evaluation")
print("-" * 40)

print(
    f"Silhouette Score: "
    f"{silhouette:.6f}"
)

print(
    f"Davies-Bouldin Index: "
    f"{davies_bouldin:.6f}"
)

print("\nCluster Distribution:")
print(evaluation_summary)

Customer Segmentation Evaluation
----------------------------------------
Silhouette Score: 0.628574
Davies-Bouldin Index: 0.628662

Cluster Distribution:
         Customer Count  Percentage
cluster                            
0                 69743       72.58
1                  2973        3.09
2                 20488       21.32
3                  2848        2.96
4                    43        0.04
5                     1        0.00


## Cluster Profile

Examine the behavioral and monetary characteristics of each cluster to determine whether small clusters represent meaningful customer segments or extreme outliers.

In [6]:
# Create a customer-level cluster profile
clustered_customer_data = training_features.copy()

clustered_customer_data["cluster"] = cluster_labels

cluster_profile = (
    clustered_customer_data
    .groupby("cluster")
    .agg(
        customer_count=("cluster", "size"),
        average_orders=("total_orders", "mean"),
        average_spent=("total_spent", "mean"),
        average_order_value=("average_order_value", "mean"),
        average_review_score=("average_review_score", "mean")
    )
    .round(2)
)

print("Customer Segment Profiles")
print("-" * 50)
print(cluster_profile)

Customer Segment Profiles
--------------------------------------------------
         customer_count  average_orders  average_spent  average_order_value  \
cluster                                                                       
0                 69743            1.00         147.92               127.48   
1                  2973            2.12         422.36               140.00   
2                 20488            1.00         184.90               140.25   
3                  2848            1.01        1545.58              1038.04   
4                    43            1.26       16472.17              2648.43   
5                     1            1.00      109312.64             13664.08   

         average_review_score  
cluster                        
0                        4.75  
1                        4.11  
2                        1.86  
3                        3.98  
4                        2.67  
5                        1.00  


## Cluster Interpretation

Translate the validated customer clusters into meaningful business segments based on their behavioral, monetary, and satisfaction characteristics.

In [7]:
cluster_interpretation = cluster_profile.copy()

cluster_interpretation["percentage"] = (
    cluster_interpretation["customer_count"]
    / len(clustered_customer_data)
    * 100
).round(2)

print("Cluster Interpretation Summary")
print("-" * 50)
print(cluster_interpretation)

Cluster Interpretation Summary
--------------------------------------------------
         customer_count  average_orders  average_spent  average_order_value  \
cluster                                                                       
0                 69743            1.00         147.92               127.48   
1                  2973            2.12         422.36               140.00   
2                 20488            1.00         184.90               140.25   
3                  2848            1.01        1545.58              1038.04   
4                    43            1.26       16472.17              2648.43   
5                     1            1.00      109312.64             13664.08   

         average_review_score  percentage  
cluster                                    
0                        4.75       72.58  
1                        4.11        3.09  
2                        1.86       21.32  
3                        3.98        2.96  
4                    

## Business Segment Interpretation

Translate the numerical cluster profiles into meaningful customer segments and define the business significance of each segment.

In [8]:
segment_definitions = {
    0: {
        "segment_name": "Core Satisfied Customers",
        "business_profile": "Large customer group with low purchase frequency, moderate spending, and high satisfaction.",
        "recommended_action": "Focus on retention and repeat-purchase campaigns."
    },
    1: {
        "segment_name": "High-Value Repeat Customers",
        "business_profile": "Customers with higher purchase frequency and spending than the core customer group.",
        "recommended_action": "Use loyalty programs and personalized offers to strengthen retention."
    },
    2: {
        "segment_name": "Low-Satisfaction Customers",
        "business_profile": "Large customer group with low purchase frequency and substantially lower review scores.",
        "recommended_action": "Investigate service and delivery issues and prioritize satisfaction improvement."
    },
    3: {
        "segment_name": "Premium High-Value Customers",
        "business_profile": "Small group with substantially higher spending and average order value.",
        "recommended_action": "Prioritize premium retention and high-value customer engagement."
    },
    4: {
        "segment_name": "Ultra-High-Value Niche Customers",
        "business_profile": "Very small group with exceptionally high spending and order value.",
        "recommended_action": "Provide highly personalized retention and premium customer treatment."
    },
    5: {
        "segment_name": "Extreme-Value Customer",
        "business_profile": "Single customer with exceptionally high spending and order value.",
        "recommended_action": "Treat as a special high-value case and monitor separately from broad customer segments."
    }
}

cluster_interpretation["segment_name"] = [
    segment_definitions[cluster]["segment_name"]
    for cluster in cluster_interpretation.index
]

cluster_interpretation["business_profile"] = [
    segment_definitions[cluster]["business_profile"]
    for cluster in cluster_interpretation.index
]

cluster_interpretation["recommended_action"] = [
    segment_definitions[cluster]["recommended_action"]
    for cluster in cluster_interpretation.index
]

print("Final Customer Segment Interpretation")
print("-" * 60)

print(
    cluster_interpretation[
        [
            "customer_count",
            "percentage",
            "segment_name",
            "business_profile",
            "recommended_action"
        ]
    ].to_string()
)

Final Customer Segment Interpretation
------------------------------------------------------------
         customer_count  percentage                      segment_name                                                                             business_profile                                                                       recommended_action
cluster                                                                                                                                                                                                                                                    
0                 69743       72.58          Core Satisfied Customers  Large customer group with low purchase frequency, moderate spending, and high satisfaction.                                        Focus on retention and repeat-purchase campaigns.
1                  2973        3.09       High-Value Repeat Customers          Customers with higher purchase frequency and spending than the cor

## Serialize Customer Segmentation Model

Save the complete fitted preprocessing and clustering pipeline so it can be reused by the backend for customer segmentation predictions.

In [9]:
import json
import joblib
from pathlib import Path

# Define model artifact directory
model_dir = Path(
    "../models/customer_segmentation"
)

model_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Define artifact paths
model_path = model_dir / "customer_segmentation_pipeline.joblib"
metadata_path = model_dir / "metadata.json"

# Save the complete fitted pipeline
joblib.dump(
    segmentation_pipeline,
    model_path
)

# Save model metadata
model_metadata = {
    "model_name": "Customer Segmentation",
    "algorithm": "KMeans",
    "n_clusters": 6,
    "random_state": 42,
    "n_init": 10,
    "features": segmentation_features,
    "preprocessing": [
        "Median Imputation",
        "StandardScaler"
    ],
    "silhouette_score": 0.628574,
    "davies_bouldin_index": 0.628662,
    "training_records": len(training_features)
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_metadata,
        file,
        indent=4
    )

print("Model serialization completed successfully.")
print(f"Model Path: {model_path}")
print(f"Metadata Path: {metadata_path}")
print(f"Model Size: {model_path.stat().st_size / 1024:.2f} KB")

Model serialization completed successfully.
Model Path: ..\models\customer_segmentation\customer_segmentation_pipeline.joblib
Metadata Path: ..\models\customer_segmentation\metadata.json
Model Size: 377.43 KB
